# SmartStock ML Phase 7B — Simple Machine Learning Model

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sebass369/smartstock-ml/blob/main/notebooks/phase_7b_machine_learning.ipynb)

## 1. Project and Privacy

This notebook trains the project's first machine-learning model on deterministic
synthetic data using only the nine approved public product aliases. Results are
not sales records, operational guidance, ordering guidance, or evidence of
real-world performance.

The model forecasts demand only. It does not change the Phase 7A contract, the
Phase 4B ordering policy, or inventory simulation, and it writes no file.

## 2. Reproducible Colab Setup

A fresh Colab runtime clones the public repository and checks out an immutable
commit revision rather than a mutable branch, so this notebook always runs
against the exact source it was validated with. The install adds the project
with its `analysis` and `ml` optional dependency groups. Local uploads, Google
Drive, external datasets, and credentials are not required.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = "https://github.com/sebass369/smartstock-ml.git"
SOURCE_REVISION = "0bd4500cf728bd2cc0405073566290297d640457"

try:
    import google.colab  # type: ignore[import-not-found]  # noqa: F401
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True

if IN_COLAB:
    project_root = Path("/content/smartstock-ml")
    if project_root.exists():
        raise RuntimeError(
            "Expected a fresh Colab runtime; restart and run all cells."
        )
    subprocess.run(
        ["git", "clone", "--quiet", REPOSITORY_URL, str(project_root)],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(project_root), "checkout", "--quiet", SOURCE_REVISION],
        check=True,
    )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            f"{project_root}[analysis,ml]",
        ],
        check=True,
    )
else:
    project_root = Path.cwd()
    if not (project_root / "PROJECT.md").is_file():
        raise RuntimeError(
            "Run this notebook from the SmartStock ML repository root."
        )

print(f"Project root: {project_root}")
print(f"Source revision: {SOURCE_REVISION}")

## 3. Generate Baseline, Candidate, and Model Forecasts

The notebook calls reusable package functions. Demand generation, training-row
selection, the model pipeline, prediction clipping, validation, and metric
calculations all live in `src/smartstock/`.

In [ ]:
from datetime import date

import matplotlib.pyplot as plt
import pandas as pd

from smartstock.config import (
    load_delivery_config,
    load_generation_config,
    load_products_config,
)
from smartstock.generator import generate_daily_records
from smartstock.ml_forecasting import (
    build_training_rows,
    fit_demand_model,
    generate_ml_cycle_forecast_records,
    summarize_ml_forecasts,
)

DEFAULT_SEED = 42
CONFIG_ROOT = project_root / "config"
products_config = load_products_config(CONFIG_ROOT / "products.yaml")
delivery_config = load_delivery_config(CONFIG_ROOT / "delivery.yaml")
generation_config = load_generation_config(CONFIG_ROOT / "generation.yaml")
product_order = [product["product_id"] for product in products_config["products"]]

demand_records = generate_daily_records(
    products_config,
    delivery_config,
    generation_config,
    seed=DEFAULT_SEED,
)
ml_records = generate_ml_cycle_forecast_records(
    demand_records,
    products_config,
    delivery_config,
)
ml_summary = summarize_ml_forecasts(ml_records, products_config)

## 4. Validate the Evaluation Contract

One 14-day warm-up cycle leaves three complete forecast origins. Each origin
contains all nine products, for 27 records in stable origin and product order.
The model is refitted independently at each origin, so `training_row_count`
grows from 126 to 252 to 378 rows that are all strictly before their origin.

In [ ]:
ml_frame = pd.DataFrame(ml_records)
for column in (
    "forecast_origin_date",
    "training_start_date",
    "training_end_date",
    "target_start_date",
    "target_end_date",
):
    ml_frame[column] = pd.to_datetime(ml_frame[column])
ml_frame["product_id"] = pd.Categorical(
    ml_frame["product_id"],
    categories=product_order,
    ordered=True,
)

assert len(ml_frame) == 27
assert ml_frame["product_id"].nunique() == 9
assert ml_frame["forecast_origin_date"].nunique() == 3
assert sorted(ml_frame["training_row_count"].unique()) == [126, 252, 378]
assert (ml_frame["training_end_date"] < ml_frame["target_start_date"]).all()
ml_frame.head(9)

## 5. The Trained Model and Its Coefficients

The pipeline has two steps and no scaling step. Step one turns the product name
and the weekday into indicator columns, using category lists taken from project
configuration so the columns are always in the same order. Step two fits
ordinary least squares, which adds a baseline number, a per-product adjustment,
and a per-weekday adjustment to estimate one day's demand.

The model uses exactly two inputs, and both are known on the delivery day
before demand is observed: the approved product alias and the target calendar
weekday. `is_high_demand_day` is excluded. It is known in advance, so this is
not a leakage decision — it is excluded because it exposes a synthetic
generator rule, because `high_demand_days` is still provisional for five
products, and because it would make this demonstration artificially favorable.

Every coefficient below is an adjustment in demand units per day, measured
against the reference product and the reference weekday that the encoder drops.

In [ ]:
final_origin = date.fromisoformat(
    max(record["forecast_origin_date"] for record in ml_records)
)
final_training_rows = build_training_rows(demand_records, final_origin)
final_model = fit_demand_model(final_training_rows)

feature_names = final_model.named_steps["features"].get_feature_names_out()
coefficients = final_model.named_steps["regressor"].coef_
coefficient_frame = pd.DataFrame(
    [
        {
            "input": "product_id" if "__x0_" in str(name) else "weekday",
            "level": str(name).split("_", 3)[-1],
            "units_per_day": float(value),
        }
        for name, value in zip(feature_names, coefficients)
    ]
)

print(f"Final forecast origin: {final_origin.isoformat()}")
print(f"Training rows: {len(final_training_rows)}")
baseline_intercept = float(final_model.named_steps["regressor"].intercept_)
print(f"Baseline intercept: {baseline_intercept:.4f}")
coefficient_frame.round(4)

## 6. Three-Method Comparison

All three methods are scored on the same 27 product-cycle records with the same
actual values, using the existing Phase 7A metric functions. MAE stays in demand
units and WAPE is a percentage. Predictions are not rounded before evaluation.

**The trained model is expected to match the expanding weekday-mean candidate
and may lose to the previous-cycle baseline. That is a valid and informative
result, not a defect.** The reason is arithmetic. The first synthetic date is a
Tuesday, every training window is 14, 28, or 42 days, and every target window is
14 days, so each window contains each weekday an equal number of times. Least
squares makes the residuals orthogonal to the product indicator columns, which
forces the average fitted value over a product's training rows to equal that
product's observed average. Summing the 14 target-day estimates therefore
returns exactly 14 times the product's pre-origin mean demand — the same
quantity the Phase 7A candidate already computes a different way.

In [ ]:
overall_metrics = pd.DataFrame(
    [
        {
            "method": label,
            "mae_units": ml_summary["overall"][f"{key}_mae_units"],
            "wape_percent": ml_summary["overall"][f"{key}_wape_percent"],
        }
        for key, label in (
            ("baseline", "Previous completed cycle baseline"),
            ("candidate", "Expanding weekday mean candidate"),
            ("model", "Trained linear model"),
        )
    ]
)
print("Model versus baseline:", ml_summary["model_vs_baseline_result_counts"])
print("Model versus candidate:", ml_summary["model_vs_candidate_result_counts"])
overall_metrics.round(6)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 11), sharex=True)
for axis, product_id in zip(axes.flat, product_order, strict=True):
    product_frame = ml_frame.loc[ml_frame["product_id"] == product_id]
    axis.plot(
        product_frame["forecast_origin_date"],
        product_frame["actual_demand_units"],
        marker="o",
        label="Actual",
    )
    axis.plot(
        product_frame["forecast_origin_date"],
        product_frame["baseline_prediction_units"],
        marker="s",
        label="Baseline",
    )
    axis.plot(
        product_frame["forecast_origin_date"],
        product_frame["candidate_prediction_units"],
        marker="^",
        label="Candidate",
    )
    axis.plot(
        product_frame["forecast_origin_date"],
        product_frame["model_prediction_units"],
        marker="x",
        linestyle="--",
        label="Trained model",
    )
    axis.set_title(product_id)
    axis.grid(alpha=0.25)
    axis.tick_params(axis="x", rotation=35)
for axis in axes[:, 0]:
    axis.set_ylabel("14-day demand units")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=4)
fig.suptitle("Synthetic 14-Day Demand: Actual, Phase 7A Methods, and Model", y=1.02)
fig.tight_layout()
plt.show()

## 7. Per-Product Error Comparison

MAE remains in demand units. Product-level results use the approved absolute tie
tolerance of `1e-9`, which is why the model and candidate register as ties even
though their cycle totals differ by a few units in the last bits of floating
point. Each panel rests on only three synthetic evaluation cycles, so these
per-product differences carry very little statistical weight.

In [ ]:
product_metrics = pd.DataFrame(
    [
        {"product_id": product_id, **ml_summary["by_product"][product_id]}
        for product_id in product_order
    ]
)
positions = range(len(product_order))
fig, axis = plt.subplots(figsize=(13, 6))
for offset, column, label in (
    (-0.26, "baseline_mae_units", "Baseline MAE"),
    (0.0, "candidate_mae_units", "Candidate MAE"),
    (0.26, "model_mae_units", "Trained model MAE"),
):
    axis.bar(
        [position + offset for position in positions],
        product_metrics[column],
        width=0.26,
        label=label,
    )
axis.set_xticks(list(positions), product_order, rotation=40, ha="right")
axis.set_ylabel("Mean absolute error in demand units")
axis.set_title("Baseline, Candidate, and Trained Model Error by Product")
axis.legend()
axis.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()
product_metrics[
    [
        "product_id",
        "baseline_mae_units",
        "candidate_mae_units",
        "model_mae_units",
        "model_vs_baseline_result",
        "model_vs_candidate_result",
    ]
].round(6)

## 8. Limitations

- The dataset contains only 56 deterministic synthetic days and three
  evaluation cycles.
- The 126, 252, and 378 training rows rest on only 14, 28, and 42 distinct
  dates. Weekday effects at the first origin come from two dates per weekday, so
  the row count overstates the available evidence.
- The trained model is expected to match the expanding weekday-mean candidate
  and may lose to the previous-cycle baseline. This is a valid and informative
  result about short, weekday-balanced synthetic data, not a defect.
- The generator applies a configured `+1` adjustment on each product's
  high-demand weekdays, so the model is learning project-authored rules rather
  than customer behavior.
- The generator draws demand independently for each product, so there are no
  shared daily shocks. Real retail demand would not behave this way.
- `is_high_demand_day`, lagged demand, rolling statistics, inventory outcomes,
  stockouts, waste, and ordering recommendations are all excluded as model
  inputs.
- Ridge regression is not used. Three evaluation origins cannot support
  leakage-safe selection of a regularization strength.
- `LinearRegression` uses no randomness, so no model random seed exists. Seed 42
  seeds the synthetic data generator only.
- No accuracy threshold or combined score is defined, and no real-world forecast
  accuracy is claimed.
- Forecasts are not fed into ordering, inventory simulation, optimization,
  deployment, an API, or a dashboard. No model file and no prediction CSV are
  written.